# 🏀 March Machine Learning Mania 2026: Advanced Elo & Meta-Ensemble Pipeline


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### 📖 1. Methodology & Approach
To predict tournament upsets and accurately model team strengths, simple win rates are not enough. This notebook implements a professional sports analytics workflow:

1. **Dynamic Elo Ratings:** We calculate season-long running Elo ratings for every team. Elo naturally captures strength of schedule and late-season momentum better than static win percentages.

2. **Advanced Metrics (Four Factors):** We use Detailed Results to calculate True Shooting Percentage (TS%) and Rebound Margins, simulating Dean Oliver's "Four Factors" of basketball success.

3. **Calibrated Meta-Ensemble:** We train both **LightGBM** and **XGBoost** models on a symmetric dataset, averaging their probabilities to minimize the Brier Score (Kaggle's evaluation metric).

### 🛠️ 2. Environment Setup & Visuals

In [ ]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss, log_loss
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')

# Advanced Aesthetic Styling
plt.style.use('fivethirtyeight')
sns.set_theme(style="whitegrid", palette="mako")
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12})

### 📂 3. Loading the Dataset & Massey Ordinals

In [ ]:
DATA_PATH = '/kaggle/input/march-machine-learning-mania-2026/'

# Load Detailed Regular Season Data (for advanced stats)
m_reg_detailed = pd.read_csv(DATA_PATH + 'MRegularSeasonDetailedResults.csv')
w_reg_detailed = pd.read_csv(DATA_PATH + 'WRegularSeasonDetailedResults.csv')

# Load Tournament Results & Seeds
m_tourney = pd.read_csv(DATA_PATH + 'MNCAATourneyCompactResults.csv')
w_tourney = pd.read_csv(DATA_PATH + 'WNCAATourneyCompactResults.csv')
m_seeds = pd.read_csv(DATA_PATH + 'MNCAATourneySeeds.csv')
w_seeds = pd.read_csv(DATA_PATH + 'WNCAATourneySeeds.csv')

# Load Massey Ordinals (Men's Consensus Rankings)
# Fix: Changed 'DayNum' to 'RankingDayNum' 
m_massey = pd.read_csv(DATA_PATH + 'MMasseyOrdinals.csv')
m_massey = m_massey[m_massey['RankingDayNum'] >= 128].groupby(['Season', 'TeamID'])['OrdinalRank'].mean().reset_index()

# Load Stage 2 Submission File
sub_file = 'SampleSubmissionStage2.csv' if os.path.exists(DATA_PATH + 'SampleSubmissionStage2.csv') else 'SampleSubmissionStage1.csv'
sub_df = pd.read_csv(DATA_PATH + sub_file)

### 🧮 4. Advanced Feature Engineering: Elo & Efficiency
 Here we define the core engine of our model. We calculate the Expected Score $E_A$ for Team A against Team B using the standard Elo formula:$$E_A = \frac{1}{1 + 10^{(R_B - R_A) / 400}}$$Where $R_A$ and $R_B$ are the current ratings of the two teams.

In [ ]:
def extract_seed(seed_str):
    return int(''.join(filter(str.isdigit, str(seed_str))))

def calculate_advanced_stats_and_elo(detailed_df, seeds_df):
    """Calculates True Shooting %, Rebound Margin, and Season-End Elo Ratings"""
    
    # 1. Advanced Metrics Calculation
    df = detailed_df.copy()
    
    # True Shooting Percentage: Points / (2 * (FGA + 0.44 * FTA))
    df['W_TS'] = df['WScore'] / (2 * (df['WFGA'] + 0.44 * df['WFTA']))
    df['L_TS'] = df['LScore'] / (2 * (df['LFGA'] + 0.44 * df['LFTA']))
    
    # Rebound Margin
    df['W_RebMargin'] = (df['WOR'] + df['WDR']) - (df['LOR'] + df['LDR'])
    df['L_RebMargin'] = -df['W_RebMargin']
    
    # Aggregate Stats
    w_stats = df.groupby(['Season', 'WTeamID']).agg(
        TS_mean=('W_TS', 'mean'), Reb_mean=('W_RebMargin', 'mean'), Wins=('WScore', 'count')
    ).reset_index().rename(columns={'WTeamID': 'TeamID'})
    
    l_stats = df.groupby(['Season', 'LTeamID']).agg(
        TS_mean=('L_TS', 'mean'), Reb_mean=('L_RebMargin', 'mean'), Losses=('LScore', 'count')
    ).reset_index().rename(columns={'LTeamID': 'TeamID'})
    
    # Combine Wins and Losses
    season_stats = pd.merge(w_stats, l_stats, on=['Season', 'TeamID'], how='outer').fillna(0)
    season_stats['TS_Pct'] = (season_stats['TS_mean_x'] * season_stats['Wins'] + season_stats['TS_mean_y'] * season_stats['Losses']) / (season_stats['Wins'] + season_stats['Losses'])
    season_stats['Reb_Margin'] = (season_stats['Reb_mean_x'] * season_stats['Wins'] + season_stats['Reb_mean_y'] * season_stats['Losses']) / (season_stats['Wins'] + season_stats['Losses'])
    season_stats['WinRate'] = season_stats['Wins'] / (season_stats['Wins'] + season_stats['Losses'])
    
    # 2. Elo Rating System Simulation
    elo_ratings = {}
    final_elos = []
    
    for row in df.itertuples():
        season, w_team, l_team = row.Season, row.WTeamID, row.LTeamID
        
        # Initialize Elo if team not seen this season
        w_key, l_key = f"{season}_{w_team}", f"{season}_{l_team}"
        if w_key not in elo_ratings: elo_ratings[w_key] = 1500
        if l_key not in elo_ratings: elo_ratings[l_key] = 1500
            
        r_w, r_l = elo_ratings[w_key], elo_ratings[l_key]
        
        # Expected Win Probabilities
        e_w = 1 / (1 + 10 ** ((r_l - r_w) / 400))
        
        # Update Ratings (K-factor = 20)
        K = 20 
        elo_ratings[w_key] += K * (1 - e_w)
        elo_ratings[l_key] += K * (0 - (1 - e_w))
        
    for key, elo in elo_ratings.items():
        s, t = key.split('_')
        final_elos.append({'Season': int(s), 'TeamID': int(t), 'Elo': elo})
        
    elo_df = pd.DataFrame(final_elos)
    
    # 3. Merge All Features
    features = season_stats[['Season', 'TeamID', 'WinRate', 'TS_Pct', 'Reb_Margin']]
    features = features.merge(elo_df, on=['Season', 'TeamID'], how='left')
    
    seeds_df['SeedNum'] = seeds_df['Seed'].apply(extract_seed)
    features = features.merge(seeds_df[['Season', 'TeamID', 'SeedNum']], on=['Season', 'TeamID'], how='left').fillna(14)
    
    return features

print("Engineered advanced features (Elo, TS%, Rebounds) for Men & Women...")
m_features = calculate_advanced_stats_and_elo(m_reg_detailed, m_seeds)
w_features = calculate_advanced_stats_and_elo(w_reg_detailed, w_seeds)

# Merge Massey Ordinals to Men's Features
m_features = m_features.merge(m_massey, on=['Season', 'TeamID'], how='left')
m_features['OrdinalRank'] = m_features['OrdinalRank'].fillna(m_features['OrdinalRank'].mean()) # Fill NaNs

### 🔀 5. Building the Symmetric Training Matrix

In [ ]:
def build_symmetric_training_set(tourney_data, features, is_mens=False):
    df1 = tourney_data[['Season', 'WTeamID', 'LTeamID']].copy()
    df1.columns = ['Season', 'Team1', 'Team2']
    df1['Target'] = 1
    
    df2 = tourney_data[['Season', 'LTeamID', 'WTeamID']].copy()
    df2.columns = ['Season', 'Team1', 'Team2']
    df2['Target'] = 0
    
    train = pd.concat([df1, df2]).reset_index(drop=True)
    
    # Merge Team 1
    train = train.merge(features, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], how='left').drop('TeamID', axis=1)
    train.rename(columns={c: f"{c}_T1" for c in features.columns if c not in ['Season', 'TeamID']}, inplace=True)
    
    # Merge Team 2
    train = train.merge(features, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'], how='left').drop('TeamID', axis=1)
    train.rename(columns={c: f"{c}_T2" for c in features.columns if c not in ['Season', 'TeamID']}, inplace=True)
    
    # Calculate Deltas (The engine of the model)
    train['SeedDiff'] = train['SeedNum_T1'] - train['SeedNum_T2']
    train['EloDiff'] = train['Elo_T1'] - train['Elo_T2']
    train['WinRateDiff'] = train['WinRate_T1'] - train['WinRate_T2']
    train['TS_PctDiff'] = train['TS_Pct_T1'] - train['TS_Pct_T2']
    train['Reb_MarginDiff'] = train['Reb_Margin_T1'] - train['Reb_Margin_T2']
    
    if is_mens:
        train['RankDiff'] = train['OrdinalRank_T1'] - train['OrdinalRank_T2']
        
    return train

m_train = build_symmetric_training_set(m_tourney, m_features, is_mens=True)
w_train = build_symmetric_training_set(w_tourney, w_features, is_mens=False)

# Define predictive columns
base_cols = ['SeedDiff', 'EloDiff', 'WinRateDiff', 'TS_PctDiff', 'Reb_MarginDiff', 
             'Elo_T1', 'Elo_T2', 'SeedNum_T1', 'SeedNum_T2']
m_cols = base_cols + ['RankDiff']
w_cols = base_cols

### 🤖 6. Calibrated Meta-Ensemble Training
We train a blend of XGBoost and LightGBM. Tree-based models are excellent at finding non-linear interactions (e.g., when a high Elo difference matters more or less depending on the Rebound Margin).

In [ ]:
def train_ensemble(train_df, feature_cols, title_name):
    print(f"\nTraining Ensembles for {title_name}...")
    X = train_df[feature_cols]
    y = train_df['Target']
    
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
    oof_preds = np.zeros(len(X))
    lgb_models, xgb_models = [], []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        # 1. LightGBM Classifier
        lgb_mod = lgb.LGBMClassifier(
            n_estimators=800, learning_rate=0.03, max_depth=4, 
            subsample=0.8, colsample_bytree=0.8, random_state=fold, verbose=-1
        )
        lgb_mod.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
        lgb_preds = lgb_mod.predict_proba(X_va)[:, 1]
        
        # 2. XGBoost Classifier
        xgb_mod = xgb.XGBClassifier(
            n_estimators=800, learning_rate=0.03, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, random_state=fold, 
            eval_metric='logloss', early_stopping_rounds=50
        )
        xgb_mod.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        xgb_preds = xgb_mod.predict_proba(X_va)[:, 1]
        
        # Blend probabilities
        blended_preds = (lgb_preds * 0.5) + (xgb_preds * 0.5)
        oof_preds[val_idx] = blended_preds
        
        lgb_models.append(lgb_mod)
        xgb_models.append(xgb_mod)
        
    brier = brier_score_loss(y, oof_preds)
    ll = log_loss(y, oof_preds)
    print(f"[{title_name}] Out-Of-Fold Brier Score: {brier:.5f} | Log Loss: {ll:.5f}")
    
    return lgb_models, xgb_models, oof_preds

m_lgb, m_xgb, m_oof = train_ensemble(m_train, m_cols, "Men's Tournament")
w_lgb, w_xgb, w_oof = train_ensemble(w_train, w_cols, "Women's Tournament")

### 📊 7. Visualizing The Findings
Let's look at what our models learned. Elo Difference and Seed Difference usually dominate, but advanced efficiency metrics offer edge cases for upsets!

In [ ]:
# 1. Feature Importance (Using LightGBM fold 1 as proxy)
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

lgb.plot_importance(m_lgb[0], ax=ax[0], title="Men's Model Feature Importance", 
                    importance_type='gain', max_num_features=10, height=0.6, color='royalblue')
lgb.plot_importance(w_lgb[0], ax=ax[1], title="Women's Model Feature Importance", 
                    importance_type='gain', max_num_features=10, height=0.6, color='crimson')
plt.tight_layout()
plt.show()

# 2. Prediction Confidence Distribution
plt.figure(figsize=(10, 5))
sns.histplot(m_oof, bins=50, color='royalblue', kde=True, label="Men's Preds", alpha=0.5, stat='density')
sns.histplot(w_oof, bins=50, color='crimson', kde=True, label="Women's Preds", alpha=0.5, stat='density')
plt.title("OOF Predicted Probabilities (Symmetric 50/50 Target)", fontweight='bold')
plt.xlabel("Predicted Probability of Team 1 Winning")
plt.legend()
plt.show()

### 📝 8. Inferencing the Stage File & Submission

In [ ]:
def inference(sub_df, features, lgb_mods, xgb_mods, feature_cols, is_mens=False):
    df = sub_df.copy()
    df['Season'] = df['ID'].apply(lambda x: int(x.split('_')[0]))
    df['Team1'] = df['ID'].apply(lambda x: int(x.split('_')[1]))
    df['Team2'] = df['ID'].apply(lambda x: int(x.split('_')[2]))
    
    # Merge T1 & T2
    df = df.merge(features, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], how='left').drop('TeamID', axis=1)
    df.rename(columns={c: f"{c}_T1" for c in features.columns if c not in ['Season', 'TeamID']}, inplace=True)
    
    df = df.merge(features, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'], how='left').drop('TeamID', axis=1)
    df.rename(columns={c: f"{c}_T2" for c in features.columns if c not in ['Season', 'TeamID']}, inplace=True)
    
    # Deltas
    df['SeedDiff'] = df['SeedNum_T1'] - df['SeedNum_T2']
    df['EloDiff'] = df['Elo_T1'] - df['Elo_T2']
    df['WinRateDiff'] = df['WinRate_T1'] - df['WinRate_T2']
    df['TS_PctDiff'] = df['TS_Pct_T1'] - df['TS_Pct_T2']
    df['Reb_MarginDiff'] = df['Reb_Margin_T1'] - df['Reb_Margin_T2']
    if is_mens: df['RankDiff'] = df['OrdinalRank_T1'] - df['OrdinalRank_T2']
        
    X = df[feature_cols]
    
    # Average across all Folds AND both model types (Meta-Ensembling)
    final_preds = np.zeros(len(X))
    for l_mod, x_mod in zip(lgb_mods, xgb_mods):
        final_preds += (l_mod.predict_proba(X)[:, 1] * 0.5) / 5
        final_preds += (x_mod.predict_proba(X)[:, 1] * 0.5) / 5
        
    df['Pred'] = np.clip(final_preds, 0.01, 0.99) # Clipping for Log Loss safety
    return df[['ID', 'Pred']]

# Kaggle Convention: Men's Teams < 3000, Women's Teams >= 3000
m_mask = sub_df['ID'].apply(lambda x: int(x.split('_')[1])) < 3000
m_sub = sub_df[m_mask].copy()
w_sub = sub_df[~m_mask].copy()

print("\nGenerating final predictions...")
m_preds = inference(m_sub, m_features, m_lgb, m_xgb, m_cols, is_mens=True)
w_preds = inference(w_sub, w_features, w_lgb, w_xgb, w_cols, is_mens=False)

# Compile submission
final_submission = pd.concat([m_preds, w_preds]).sort_values('ID').reset_index(drop=True)
final_submission.to_csv('submission.csv', index=False)

print("✅ Pipeline Complete! Output saved to 'submission.csv'")
display(final_submission.head())